## Nightly Report: IQ, AOS, Thermal, Aberrations and Degrees of Freedom

Owners: **Guillem Megias** , **Craig Lage** <br>
Last Verified to Run: **2026-03-05** <br>

In [ ]:
# Times Square Parameters
day_obs = 20260305
seq_min = 0
seq_max = 1500
make_detail_plot = True
detailed_seq_min = 300
detailed_seq_max = 350

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.dates import DateFormatter
from matplotlib.patches import Rectangle, Patch
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

In [ ]:
import galsim
import numpy as np
import pandas as pd
pd.set_option('future.no_silent_downcasting', True)

def getPsfGradPerZernike(
    diameter: float = 8.36,
    obscuration: float = 0.612,
    jmin: int = 4,
    jmax: int = 22,
) -> np.ndarray:
    """Get the gradient of the PSF FWHM with respect to each Zernike.

    This function takes no positional arguments. All parameters must be passed
    by name (see the list of parameters below).

    Parameters
    ----------
    diameter : float, optional
        The diameter of the telescope aperture, in meters.
        (the default, 8.36, corresponds to the LSST primary mirror)
    obscuration : float, optional
        Central obscuration of telescope aperture (i.e. R_outer / R_inner).
        (the default, 0.612, corresponds to the LSST primary mirror)
    jmin : int, optional
        The minimum Noll index, inclusive. Must be >= 0. (the default is 4)
    jmax : int, optional
        The max Zernike Noll index, inclusive. Must be >= jmin.
        (the default is 22.)

    Returns
    -------
    np.ndarray
        Gradient of the PSF FWHM with respect to the corresponding Zernike.
        Units are arcsec / micron.

    Raises
    ------
    ValueError
        If jmin is negative or jmax is less than jmin
    """
    # Check jmin and jmax
    if jmin < 0:
        raise ValueError("jmin cannot be negative.")
    if jmax < jmin:
        raise ValueError("jmax must be greater than jmin.")

    # Calculate the conversion factors
    conversion_factors = np.zeros(jmax + 1)
    for i in range(jmin, jmax + 1):
        # Set coefficients for this Noll index: coefs = [0, 0, ..., 1]
        # Note the first coefficient is Noll index 0, which does not exist and
        # is therefore always ignored by galsim
        coefs = [0] * i + [1]

        # Create the Zernike polynomial with these coefficients
        R_outer = diameter / 2
        R_inner = R_outer * obscuration
        Z = galsim.zernike.Zernike(coefs, R_outer=R_outer, R_inner=R_inner)

        # We can calculate the size of the PSF from the RMS of the gradient of
        # the wavefront. The gradient of the wavefront perturbs photon paths.
        # The RMS quantifies the size of the collective perturbation.
        # If we expand the wavefront gradient in another series of Zernike
        # polynomials, we can exploit the orthonormality of the Zernikes to
        # calculate the RMS from the Zernike coefficients.
        rms_tilt = np.sqrt(np.sum(Z.gradX.coef**2 + Z.gradY.coef**2) / 2)

        # Convert to arcsec per micron
        rms_tilt = np.rad2deg(rms_tilt * 1e-6) * 3600

        # Convert rms -> fwhm
        fwhm_tilt = 2 * np.sqrt(2 * np.log(2)) * rms_tilt

        # Save this conversion factor
        conversion_factors[i] = fwhm_tilt

    return conversion_factors[jmin:]


def convertZernikesToPsfWidth(
    zernikes: np.ndarray,
    diameter: float = 8.36,
    obscuration: float = 0.612,
    jmin: int = 4,
) -> np.ndarray:
    """Convert Zernike amplitudes to quadrature contribution to the PSF FWHM.

    Parameters
    ----------
    zernikes : np.ndarray
        Zernike amplitudes (in microns), starting with Noll index `jmin`.
        Either a 1D array of zernike amplitudes, or a 2D array, where each row
        corresponds to a different set of amplitudes.
    diameter : float
        The diameter of the telescope aperture, in meters.
        (the default, 8.36, corresponds to the LSST primary mirror)
    obscuration : float
        Central obscuration of telescope aperture (i.e. R_outer / R_inner).
        (the default, 0.612, corresponds to the LSST primary mirror)
    jmin : int
        The minimum Zernike Noll index, inclusive. Must be >= 0. The
        max Noll index is inferred from `jmin` and the length of `zernikes`.
        (the default is 4, which ignores piston, x & y offsets, and tilt.)

    Returns
    -------
    dFWHM: np.ndarray
        Quadrature contribution of each Zernike vector to the PSF FWHM
        (in arcseconds).

    Notes
    -----
    Converting Zernike amplitudes to their quadrature contributions to the PSF
    FWHM allows for easier physical interpretation of Zernike amplitudes and
    the performance of the AOS system.

    For example, image we have a true set of zernikes, [Z4, Z5, Z6], such that
    ConvertZernikesToPsfWidth([Z4, Z5, Z6]) = [0.1, -0.2, 0.3] arcsecs.
    These Zernike perturbations increase the PSF FWHM by
    sqrt[(0.1)^2 + (-0.2)^2 + (0.3)^2] ~ 0.37 arcsecs.

    If the AOS perfectly corrects for these perturbations, the PSF FWHM will
    not increase in size. However, imagine the AOS estimates zernikes, such
    that ConvertZernikesToPsfWidth([Z4, Z5, Z6]) = [0.1, -0.3, 0.4] arcsecs.
    These estimated Zernikes, do not exactly match the true Zernikes above.
    Therefore, the post-correction PSF will still be degraded with respect to
    the optimal PSF. In particular, the PSF FWHM will be increased by
    sqrt[(0.1 - 0.1)^2 + (-0.2 - (-0.3))^2 + (0.3 - 0.4)^2] ~ 0.14 arcsecs.

    This conversion depends on a linear approximation that begins to break down
    for RSS(dFWHM) > 0.20 arcsecs. Beyond this point, the approximation tends
    to overestimate the PSF degradation. In other words, if
    sqrt(sum( dFWHM^2 )) > 0.20 arcsec, it is likely that dFWHM is
    over-estimated. However, the point beyond which this breakdown begins
    (and whether the approximation over- or under-estimates dFWHM) can change,
    depending on which Zernikes have large amplitudes. In general, if you have
    large Zernike amplitudes, proceed with caution!
    Note that if the amplitudes Z_est and Z_true are large, this is okay, as
    long as |Z_est - Z_true| is small.

    For a notebook demonstrating where the approximation breaks down:
    https://gist.github.com/jfcrenshaw/24056516cfa3ce0237e39507674a43e1

    Raises
    ------
    ValueError
        If jmin is negative
    """
    # Check jmin
    if jmin < 0:
        raise ValueError("jmin cannot be negative.")

    # Calculate jmax from jmin and the length of the zernike array
    jmax = jmin + np.array(zernikes).shape[-1] - 1

    # Calculate the conversion factors for each zernike
    conversion_factors = getPsfGradPerZernike(
        jmin=jmin,
        jmax=jmax,
        diameter=diameter,
        obscuration=obscuration,
    )

    # Convert the Zernike amplitudes from microns to their quadrature
    # contribution to the PSF FWHM
    dFWHM = conversion_factors * zernikes

    return dFWHM

band_colors = {
    "u": "#0c71ff",
    "g": "#49be61",
    "r": "#c61c00",
    "i": "#ffc200",
    "z": "#f341a2",
    "y": "#5d0000",
}

def annotate_bands(data: pd.DataFrame, ax: plt.Axes, bottom_stripes=False) -> None:
    """Fill an ax with band colors

    Parameters
    ----------
    data : pd.DataFrame
        Table of data that is being plotted
    ax : plt.Axes
        Axis on which to add annotations
    bottom_stripes : Bool
        Decides whether to put a full color stripe at the bottom
        or a light color strip on the whole Axes
    """
    # Get the axis limits
    xlim = ax.get_xlim()
    ylim = ax.get_ylim()

    maxSeq = np.max(data['seq'].values)
    change_seqs = data.index[data['band'].ne(data['band'].shift())].to_list()
    change_intervals = []
    stripe_colors = []    
    for i, seq in enumerate(change_seqs):
        if i == len(change_seqs) - 1:
            break
        change_intervals.append((seq, change_seqs[i + 1] - seq))
        band_name = list(data['band'].loc[seq])[0]
        stripe_colors.append(band_colors[band_name])
    change_intervals.append((change_seqs[-1], maxSeq - change_seqs[-1]))
    band_name = list(data['band'].loc[maxSeq])[0]
    stripe_colors.append(band_colors[band_name])
    
    if bottom_stripes:
        ax.broken_barh(change_intervals, (ylim[0], 0.08 * (ylim[1]-ylim[0])), facecolors=stripe_colors)
        # Add the lgend
        unique_bands = data['band'].str[0].unique()  # Get first character of each band
        for band_char in sorted(unique_bands):
            ax.scatter([], [], 
                      c=band_colors[band_char], 
                      s=100, 
                      marker='s', 
                      label=f'{band_char}')
    else:
        ax.broken_barh(change_intervals, (ylim[0], ylim[1]-ylim[0]), facecolors=stripe_colors, alpha=0.07)


def find_blocks(data: pd.DataFrame) -> pd.DataFrame:
    """Find blocks active during the night

    Parameters
    ----------
    data : pd.DataFrame
        Table of data that is being plotted
    Returns
    -------
    changes: pd.DataFrame
        dataframe containing the sequence numbers where the block changed
        and the name of the new block
    """
    block_data = data[['seq', 'block']].sort_values(by='seq')
    change_mask = block_data['block'] != block_data['block'].shift()
    block_data['block'] = block_data['block'].str.replace('BLOCK-', '', regex=False)
    blocks = pd.DataFrame({
        'seq': block_data.loc[change_mask, 'seq'],
        'block_after': block_data['block'].loc[change_mask]
    })
    return blocks

async def find_faults(client, table):
    """Find faults during the night

    Parameters
    ----------
    client: EFD client
    table : pd.DataFrame
        Table of data that is being plotted
    Returns
    -------
    table: pd.DataFrame
        incoming dataframe with the fault events added
    """
    table_time = pd.to_datetime(table['time'], format='ISO8601', utc=True)
    topics = ['MTMount', 'MTAOS', 'MTHexapod', 'MTCamera']
    table['faults'] = [None for i in range(len(table))]
    start = Time(table['obs_start'].iloc[0], scale='tai').utc
    end = Time(table['obs_end'].iloc[-1], scale='tai').utc
    for topic in topics:
        efd_data = await client.select_time_series(f"lsst.sal.{topic}.logevent_summaryState", \
                                                ['summaryState'], \
                                                 start, end)
        if len(efd_data) == 0:
            continue
        faults = efd_data[efd_data['summaryState'] == 3]
        for i in range(len(faults)):
            fault_time = pd.to_datetime(faults.index[i], format='utc')
            closest_row = table.iloc[(table_time - fault_time).abs().argmin()]
            table.loc[table['seq'] == closest_row['seq'], 'faults'] = topic
    return table


In [ ]:
from lsst.ts.xml.tables.m1m3 import *
from lsst.ts.m1m3.utils import *


async def get_m1m3_gradients(client, data):
    """Get the M1M3 thermal gradients

    Parameters
    ----------
    client: EFD client
    data : pd.DataFrame
        Table of minimal data
    Returns
    -------
    data: pd.DataFrame
        incoming dataframe with the m1m3 temperature gradients added
    """
    date_strings = Time([str(x) for x in data['obs_start'].values], 
                        format='isot', scale='tai').utc.isot
    data_times = pd.to_datetime(date_strings,  format='ISO8601', utc=True)
    sorted_data_times = data_times.sort_values()
    start = Time(sorted_data_times[0])
    end = Time(sorted_data_times[-1])
    data_times = data_times.astype('int64')
    thermocouples = ThermocoupleAnalysis(client)
    await thermocouples.load(start, end, time_bin=30)
    gradients = thermocouples.xyz_r_gradients
    grad_times = pd.to_datetime(gradients.index, format='ISO8601', utc=True).astype('int64')
    t0 = grad_times[0]
    grad_times -= t0
    grad_times /=1E9
    data_times -= t0
    data_times /= 1E9
    names = ['x_gradient', 'y_gradient', 'z_gradient', 'radial_gradient']
    for name in names:
        values = gradients[name].values
        val_series = pd.Series(values)
        val_interpolated = val_series.interpolate()
        data[name] = np.interp(data_times, grad_times, val_interpolated)
    return data


In [ ]:
from matplotlib.patches import Patch
from matplotlib import cm

def plot_m1m3_gradients(data, ax):
    """Plot the M1M3 thermal gradients

    Parameters
    ----------
    data : pd.DataFrame
        Table of data to plot
    ax : matplotlib.axes._axes.Axes
        axes object on which to plot the data
    Returns
    -------
    """

    # --- Palettes: same hue family within groups, distinct shades per line ---
    blues   = plt.get_cmap('Blues')
    oranges = plt.get_cmap('Oranges')
    
    # Line colors (distinct, but clearly grouped)
    c_x = blues(0.75)     # darker blue
    c_y = blues(0.55)     # lighter blue
    c_r = oranges(0.75)   # darker orange
    c_z = oranges(0.55)   # lighter orange
    
    # Band colors (very light tint of the group hue)
    band_xy = blues(0.25)
    band_rz = oranges(0.25)
    
    # --- Shaded operational bands (put behind data) ---
    ax.axhspan(-0.4, 0.4, facecolor=band_xy, alpha=0.3, zorder=0)
    ax.axhspan(-0.1, 0.1, facecolor=band_rz, alpha=0.3, zorder=0)
    ax.axhline(0.4,  color=blues(0.65), linestyle="-", linewidth=1, alpha=0.5)
    ax.axhline(-0.4, color=blues(0.65), linestyle="-", linewidth=1, alpha=0.5)
    ax.axhline(0.1,  color=oranges(0.65), linestyle="-", linewidth=1, alpha=0.5)
    ax.axhline(-0.1, color=oranges(0.65), linestyle="-", linewidth=1, alpha=0.5)
    
    # --- Data lines: distinct per series with styles/markers ---
    ax.scatter(data['seq'],
        data['x_gradient'] * 8.4,
        label="X (×8.4)", color=c_x, linewidth=2.0, linestyle="-",
        s=0.5, zorder=3
    )
    ax.scatter(data['seq'],
        data['y_gradient'] * 8.4,
        label="Y (×8.4)", color=c_y, linewidth=2.0, linestyle="--",
        s=0.5, zorder=3
    )
    ax.scatter(data['seq'],
        data['radial_gradient'] * 3.7,
        label="Radial (×4.2)", color=c_r, linewidth=2.0, linestyle="-",
        s=0.5, zorder=4
    )
    ax.scatter(data['seq'],
        data['z_gradient'],
        label="Z", color=c_z, linewidth=2.0, linestyle="--",
        s=0.5, zorder=4
    )
    
    
    # --- Legends: one for data lines, one for bands (with matching colors) ---
    data_leg = ax.legend(bbox_to_anchor=(0.0, 1.25), loc='upper left',
                frameon=True, ncol=2, markerscale=4)
    ax.add_artist(data_leg)
    
    band_handles = [
        Patch(facecolor=band_xy, alpha=0.6, label="X/Y limit band (±0.4)"),
        Patch(facecolor=band_rz, alpha=0.6, label="Radial/Z limit band (±0.1)"),
    ]
    ax.legend(handles=band_handles,
              loc="upper right", frameon=True, bbox_to_anchor=(1.0, 1.25))
    
    # --- Labels, grid, cosmetics ---
    
    ax.set_ylim(-1.2, 0.6)
    return


In [ ]:
import logging

from astropy.time import Time, TimeDelta
from lsst.obs.lsst import LsstCam
from lsst.summit.utils import (
    ConsDbClient,
    getAirmassSeeingCorrection,
    getBandpassSeeingCorrection,
)
from lsst.summit.utils.efdUtils import (
    getEfdData,
    getMostRecentRowWithDataBefore,
    makeEfdClient,
)

from tqdm import tqdm as tqdm

__all__ = ["AOSDatabase"]


class AOSDatabase:
    table: pd.DataFrame

    def __init__(
        self,
        day_obs: int = 20250415,
        seq_min: int = 1,
        seq_max: int = 9999,
        consdb_url: str = "http://consdb-pq.consdb:8080/consdb",
    ) -> None:
        """Create fetcher.

        Parameters
        ----------
        seq_max : int, optional
            Maximum sequence number to fetch. Default is 9999.
        consdb_url : str, optional
            URL to create ConsDB client.
            The default is "http://consdb-pq.consdb:8080/consdb".
        """
        self.log = logging.getLogger(__name__)

        self.efd_client = makeEfdClient()
        self.cdb_client = ConsDbClient(consdb_url)

        self.det_order = list([191, 195, 199, 203])
        camera = LsstCam().getCamera()
        self.detector_names = [
            camera.get(det_id).getName() for det_id in self.det_order
        ]

        self.day_obs = day_obs
        self.seq_max = seq_max
        self.seq_min = seq_min
        self.table = pd.DataFrame()

        self.time_window = TimeDelta(0.2, format="sec")
        self.temp_time_window = TimeDelta(0.2, format="sec")

    async def create(self, simplified=False):
        self.table = await self._fetch(
            self.day_obs, self.seq_min, self.seq_max, simplified
        )

    async def update(self, simplified: bool = False) -> None:
        """Update the database by grabbing more recent exposures.
        This will only grab new sequences, not re-fetch existing ones.
        """
        # First grab new sequences
        seq_min = self.table["seq"].max() + 1
        updated_table = await self._fetch(
            self.day_obs, seq_min, self.seq_max, simplified=simplified
        )
        self.table = pd.concat(
            [self.table, updated_table],
            ignore_index=True,
        )
    
    async def _fetch(
        self, day_obs: int, seq_min: int, seq_max: int, simplified: bool = False
    ) -> pd.DataFrame:
        query = f"""
            SELECT
            e.air_temp AS air_temp,
            e.airmass AS airmass,
            e.dimm_seeing AS dimm,
            e.altitude AS elevation,
            e.azimuth AS azimuth,
            e.exposure_id AS visit_id,
            e.physical_filter as band,
            e.day_obs AS day_obs,
            e.exp_midpt AS time,
            e.dimm_seeing AS seeing,
            e.seq_num AS seq,
            e.science_program AS block,
            ccdvisit1_quicklook.psf_sigma,
            ccdvisit1_quicklook.z4,
            ccdvisit1_quicklook.z5,
            ccdvisit1_quicklook.z6,
            ccdvisit1_quicklook.z7,
            ccdvisit1_quicklook.z8,
            ccdvisit1_quicklook.z9,
            ccdvisit1_quicklook.z10,
            ccdvisit1_quicklook.z11,
            ccdvisit1_quicklook.z12,
            ccdvisit1_quicklook.z13,
            ccdvisit1_quicklook.z14,
            ccdvisit1_quicklook.z15,
            ccdvisit1_quicklook.z16,
            ccdvisit1_quicklook.z17,
            ccdvisit1_quicklook.z18,
            ccdvisit1_quicklook.z19,
            ccdvisit1_quicklook.z20,
            ccdvisit1_quicklook.z21,
            ccdvisit1_quicklook.z22,
            ccdvisit1_quicklook.z23,
            ccdvisit1_quicklook.z24,
            ccdvisit1_quicklook.z25,
            ccdvisit1_quicklook.z26,
            ccdvisit1_quicklook.z27,
            ccdvisit1_quicklook.z28,
            ccdvisit1.detector as detector,
            q.psf_sigma_median AS psf_fwhm,
            q.psf_sigma_min AS psf_fwhm_min,
            q.psf_sigma_max AS psf_fwhm_max,
            q.psf_area_median AS psf_area,
            q.psf_area_min AS psf_area_min,
            q.psf_area_max AS psf_area_max,
            q.aos_fwhm AS aos_fwhm,
            q.donut_blur_fwhm AS donut_blur_fwhm,
            q.physical_rotator_angle AS rotation_angle,
            e.obs_end,
            e.obs_start
            FROM
            cdb_lsstcam.ccdvisit1_quicklook AS ccdvisit1_quicklook,
            cdb_lsstcam.ccdvisit1 AS ccdvisit1,
            cdb_lsstcam.visit1 AS visit1,
            cdb_lsstcam.visit1_quicklook AS q,
            cdb_lsstcam.exposure AS e
            WHERE
            ccdvisit1.detector IN (191, 192, 195, 196, 199, 200, 203, 204)
            AND ccdvisit1.ccdvisit_id = ccdvisit1_quicklook.ccdvisit_id
            AND ccdvisit1.visit_id = visit1.visit_id
            AND ccdvisit1.visit_id = q.visit_id
            AND ccdvisit1.visit_id = e.exposure_id
            AND (e.img_type = 'science' or e.img_type = 'acq' or e.img_type = 'engtest')
            AND e.day_obs = {day_obs}
            AND (e.seq_num BETWEEN {seq_min} AND {seq_max})
        """
        self.table = self.cdb_client.query(query).to_pandas()
        if len(self.table) == 0:
            #raise RuntimeError("ConsDB query is empty")
            return self.table

        # Correctly declare aos_fwhm and donut_blur_fwhm as float
        self.table['psf_fwhm'] = pd.to_numeric(self.table['psf_fwhm'])
        self.table['aos_fwhm'] = pd.to_numeric(self.table['aos_fwhm'])
        self.table['donut_blur_fwhm'] = pd.to_numeric(self.table['donut_blur_fwhm'])

        # Convert PSF sigma to FWHM
        sig2fwhm = 2 * np.sqrt(2 * np.log(2))
        pixel_scale = 0.2  # arcsec / pixel
        self.table["psf_fwhm"] = self.table["psf_fwhm"] * sig2fwhm * pixel_scale
        self.table["psf_fwhm_min"] = self.table["psf_fwhm_min"] * sig2fwhm * pixel_scale
        self.table["psf_fwhm_max"] = self.table["psf_fwhm_max"] * sig2fwhm * pixel_scale
        self.table['airmass'] = np.clip(self.table['airmass'], 1.0, 3.0)

        self.table["fwhm_zenith_500nm"] = [
            fwhm
            * getAirmassSeeingCorrection(airmass)
            * getBandpassSeeingCorrection(band)
            for fwhm, band, airmass in zip(
                self.table["psf_fwhm"], self.table["band"], self.table["airmass"]
            )
        ]

        zernike_columns = [f"z{i}" for i in range(4, 29)]
        self.table["zernikes"] = self.table[zernike_columns].apply(
            lambda row: np.array(row.fillna(0.0).values, dtype=float), axis=1
        )
        self.table["zernikes_fwhm"] = self.table["zernikes"].apply(
            convertZernikesToPsfWidth
        )
        
        # Get the data for the science CCDs for fwhm_05 and fwhm_95
        visits_query = f'''
        SELECT 
        ccdvisit1_quicklook.psf_sigma,
        ccdvisit1_quicklook.psf_ixx,
        ccdvisit1_quicklook.psf_iyy,
        ccdvisit1_quicklook.psf_ixy,
        ccdvisit1.detector,
        visit1.visit_id,
        visit1.seq_num AS seq,
        visit1.day_obs,
        visit1.airmass
        FROM
        cdb_lsstcam.ccdvisit1_quicklook AS ccdvisit1_quicklook,
        cdb_lsstcam.ccdvisit1 AS ccdvisit1,
        cdb_lsstcam.visit1_quicklook AS visit1_quicklook,
        cdb_lsstcam.visit1 AS visit1 
        WHERE 
        ccdvisit1.ccdvisit_id = ccdvisit1_quicklook.ccdvisit_id
        AND ccdvisit1.visit_id = visit1.visit_id 
        AND visit1.visit_id = visit1_quicklook.visit_id
        AND ccdvisit1.detector NOT IN (168, 188, 123, 27, 0, 20, 65, 161)
        AND visit1.airmass > 0
        AND visit1.day_obs = {self.day_obs}
        AND (visit1.seq_num BETWEEN {self.seq_min} AND {self.seq_max})
        AND (visit1.img_type = 'science' or visit1.img_type = 'acq' or visit1.img_type = 'engtest')
        '''
        
        ccdvisits = self.cdb_client.query(visits_query).to_pandas()

        ccdvisits["psf_sigma"] = pd.to_numeric(ccdvisits["psf_sigma"], errors="coerce")
        ccdvisits["psf_ixx"]   = pd.to_numeric(ccdvisits["psf_ixx"], errors="coerce")
        ccdvisits["psf_iyy"]   = pd.to_numeric(ccdvisits["psf_iyy"], errors="coerce")
        ccdvisits["psf_ixy"]   = pd.to_numeric(ccdvisits["psf_ixy"], errors="coerce")

        ccdvisits["psf_fwhm"] = ccdvisits["psf_sigma"] * sig2fwhm * pixel_scale

        denom = ccdvisits["psf_ixx"] + ccdvisits["psf_iyy"]
        denom = denom.replace(0, np.nan)
        
        e1 = (ccdvisits["psf_ixx"] - ccdvisits["psf_iyy"]) / denom
        e2 = (2 * ccdvisits["psf_ixy"]) / denom
        
        ccdvisits["ellipticity"] = np.sqrt(e1.to_numpy()**2 + e2.to_numpy()**2)
        
        # --- Group by visit -------------------------------------------------------
        clean = ccdvisits.dropna(subset=["psf_fwhm", "ellipticity"])
        groups = clean.groupby("visit_id")  
        # For ellipticity, don't use the chips outside the good circle
        edge_chips = (0, 1, 2, 3, 18, 19, 20, 23, 64, 65, 68, 71, 155,
            158,160,161,185,186,187,188, 168, 169, 176, 165,
            123,124,120,117,30,33,27,28)
        ell_clean = clean[~(clean['detector'].isin(edge_chips))]
        ell_groups = ell_clean.groupby("visit_id")
        visits_summary = pd.DataFrame({
            'day_obs': groups['day_obs'].first(),
            'seq': groups['seq'].median(),
            'psf_fwhm_05': groups['psf_fwhm'].quantile(0.05),
            'psf_fwhm_95': groups['psf_fwhm'].quantile(0.95),
            'psf_fwhm_sigma': groups['psf_fwhm'].std(),
        
            # NEW — ellipticity min/median/max (true across CCDs)
            'psf_ellipticity_median': ell_groups['ellipticity'].median(),
            'psf_ellipticity_sigma': ell_groups['ellipticity'].std(),
        })
        visits_summary['psf_fwhm_95_05'] = np.sqrt(visits_summary['psf_fwhm_95']**2 - visits_summary['psf_fwhm_05']**2)
        self.table = pd.merge(
                        self.table, visits_summary, how="left", on=["seq", "day_obs"])

        self.table = await find_faults(self.efd_client, self.table)
        
        if not simplified:
            unique_day_seq = (
                self.table[["day_obs", "seq", "obs_end", "obs_start"]]
                .drop_duplicates()
                .reset_index(drop=True)
            )
            (
                days,
                seqs,
                lut,
                cam_air_temp,
                states,
                m1m3_air_temp,
                outside_temp,
                m2_air_temp,
                correction_seq,
            ) = ([] for _ in range(9))
            for idx, row in tqdm(unique_day_seq.iterrows(), total=len(unique_day_seq), disable=True):
                day_obs = int(row["day_obs"])
                seq = int(row["seq"])

                rec_end = row["obs_end"]
                rec_start = row["obs_start"]

                # ---------- Environment variables -------------
                # ----------------------------------------------
                # Get state
                # M2 and M1M3 data may be added later, but for now we 
                # just fill those DOFs with nans.
                m2_dofs_lut = np.full(20, np.nan)
                m1m3_dofs_lut = np.full(20, np.nan)
                try:
                    cam_hexapod_data = getMostRecentRowWithDataBefore(
                        self.efd_client,
                        "lsst.sal.MTHexapod.logevent_compensationOffset",
                        Time(rec_end, scale="tai").utc,
                        maxSearchNMinutes=10,
                        where=lambda df: df["salIndex"] == 1,
                    )

                    m2_hexapod_data = getMostRecentRowWithDataBefore(
                        self.efd_client,
                        "lsst.sal.MTHexapod.logevent_compensationOffset",
                        Time(rec_end, scale="tai").utc,
                        maxSearchNMinutes=10,
                        where=lambda df: df["salIndex"] == 2,
                    )

                    hexapod_val = np.array(
                        [
                            m2_hexapod_data["z"],
                            m2_hexapod_data["x"],
                            m2_hexapod_data["y"],
                            m2_hexapod_data["u"],
                            m2_hexapod_data["v"],
                            cam_hexapod_data["z"],
                            cam_hexapod_data["x"],
                            cam_hexapod_data["y"],
                            cam_hexapod_data["u"],
                            cam_hexapod_data["v"],
                        ]
                    )
                    lut_val = np.concatenate([hexapod_val, m1m3_dofs_lut, m2_dofs_lut])
                except Exception:
                    lut_val = np.full(50, np.nan)

                event = getMostRecentRowWithDataBefore(
                    self.efd_client,
                    "lsst.sal.MTAOS.logevent_degreeOfFreedom",
                    timeToLookBefore=Time(rec_start, scale="tai").utc,
                )
                out = np.empty(
                    50,
                )
                for i in range(50):
                    out[i] = event[f"aggregatedDoF{i}"]
                states_val = out

                seq_num_corr = event["visitId"]
                seq_num_corr = np.where(seq_num_corr < 10000, seq_num_corr,
                                        int(seq_num_corr - 1e5 * day_obs))
                
                # Get outside temperature
                outside_temp_data = await self.efd_client.select_time_series(
                    "lsst.sal.ESS.temperature",
                    ["temperatureItem0"],
                    Time(rec_start, scale="tai").utc,
                    Time(rec_end, scale="tai").utc + self.temp_time_window,
                    index=301,
                    convert_influx_index=True
                )
                if "temperatureItem0" in outside_temp_data:
                    outside_temp_val = outside_temp_data["temperatureItem0"].mean()
                else:
                    outside_temp_val = np.nan

                # Get M2 temperature
                m2_temp_data = await self.efd_client.select_time_series(
                    "lsst.sal.ESS.temperature",
                    ["temperatureItem0"],
                    Time(rec_start, scale="tai").utc,
                    Time(rec_end, scale="tai").utc + self.temp_time_window,
                    index=112,
                    convert_influx_index=True
                )
                if "temperatureItem0" in m2_temp_data:
                    m2_air_temp_val = m2_temp_data["temperatureItem0"].mean()
                else:
                    m2_air_temp_val = np.nan

                # Get cam temperature
                cam_temp_data = await self.efd_client.select_time_series(
                    "lsst.sal.ESS.temperature",
                    ["temperatureItem0"],
                    Time(rec_start, scale="tai").utc,
                    Time(rec_end, scale="tai").utc + self.temp_time_window,
                    index=111,
                )
                if "temperatureItem0" in cam_temp_data:
                    cam_air_temp_val = cam_temp_data["temperatureItem0"].mean()
                else:
                    cam_air_temp_val = np.nan

                # Get temperature above m1m3
                m1m3_temp_data = await self.efd_client.select_time_series(
                    "lsst.sal.ESS.temperature",
                    ["temperatureItem0"],
                    Time(rec_start, scale="tai").utc,
                    Time(rec_end, scale="tai").utc + self.temp_time_window,
                    index=113,
                    convert_influx_index=True
                )
                if "temperatureItem0" in m1m3_temp_data:
                    m1m3_air_temp_val = m1m3_temp_data["temperatureItem0"].mean()
                else:
                    m1m3_air_temp_val = np.nan

                lut.append(lut_val)
                m1m3_air_temp.append(m1m3_air_temp_val)
                cam_air_temp.append(cam_air_temp_val)
                m2_air_temp.append(m2_air_temp_val)
                outside_temp.append(outside_temp_val)
                states.append(states_val)
                days.append(day_obs)
                seqs.append(seq)
                correction_seq.append(seq_num_corr)

            # Calculate FWHM Zernike contributions
            efd_table = pd.DataFrame(
                {
                    "day_obs": days,
                    "seq": seqs,
                    "cam_air_temp": cam_air_temp,
                    "m2_air_temp": m2_air_temp,
                    "m1m3_air_temp": m1m3_air_temp,
                    "outside_temp": outside_temp,
                    "lut_state": lut,
                    "dof_state": states,
                    "seq_num_corr": correction_seq,
                }
            )

            self.table = pd.merge(
                self.table, efd_table, how="left", on=["seq", "day_obs"]
            )
            m1m3_gradient_table = await get_m1m3_gradients(self.efd_client, unique_day_seq)
            self.table = pd.merge(
                self.table, m1m3_gradient_table, how="left", on=["seq", "day_obs"]
            )
            self.table["m2_delta_t"] = (
                self.table["m2_air_temp"] - self.table["m1m3_air_temp"]
            )
            self.table["dome_delta_t"] = (
                self.table["outside_temp"] - self.table["m1m3_air_temp"]
            )
            self.table["cam_m1m3_delta_t"] = (
                self.table["cam_air_temp"] - self.table["m1m3_air_temp"]
            )

        return self.table


In [ ]:

zk_groups = [[0], [11 - 4], [1, 2], [3,4], [5, 6], [22]]
zk_group_labels = ['Z4', 'Z11', 'Z5 / Z6', 'Z7 / Z8', ' Z9 / Z10', 'Z22']

groups = [[0], [5], [1, 2], [6, 7], [3, 4], [8,9]]
group_labels = [' M2 dz', 'Cam dz', 'M2 dx/dy', 'Cam dx/dy', 'M2 tilts', 'Cam tilts']
labels = ['m2 dz', 'm2 dx', 'm2 dy', 'm2 rx', 'm2 ry',
          'cam dz', 'cam dx', 'cam dy', 'cam rx', 'cam ry']


mirror_groups = [[10, 11, 30, 31], [12, 34], [13, 14, 32, 33], [15, 16, 35, 36], [17, 18, 37, 38]]
mirror_group_labels = ['Astig', 'Spherical', 'Trefoil', 'Coma', 'Quad']
all_labels = ['M2 dz', 'M2 dx', 'M2 dy', 'M2 rx', 'M2 ry',
     'Cam dz', 'Cam dx', 'Cam dy', 'Cam rx', 'Cam ry',
     'M1M3-1', 'M1M3-2', 'M1M3-3', 'M1M3-4', 'M1M3-5',
     'M1M3-6', 'M1M3-7', 'M1M3-8', 'M1M3-9', 'M1M3-10',
     'M1M3-11', 'M1M3-12', 'M1M3-13', 'M1M3-14', 'M1M3-15',
     'M1M3-16', 'M1M3-17', 'M1M3-18', 'M1M3-19', 'M1M3-20',
     'M2-1', 'M2-2', 'M2-3', 'M2-4', 'M2-5',
     'M2-6', 'M2-7', 'M2-8', 'M2-9', 'M2-10',
     'M2-11', 'M2-12', 'M2-13', 'M2-14', 'M2-15',
     'M2-16', 'M2-17', 'M2-18', 'M2-19', 'M2-20'
     ]

vmode_labels = ['Vmode1\nM2 tilts -rx-ry', 'Vmode2\nM2 tilts -rx+ry', 
                'Vmode3\nCam tilts -rx+ry', 'Vmode4\nCam tilts rx+ry',
                'Vmode5\nZ4-Focus', 'Vmode6\nZ5-Astig-Oblique',
                'Vmode7\nZ6-Astig-Vert', 'Vmode8\nZ7-Coma-Vert',
                'Vmode9\nZ8-Come-Horiz', 'Vmode10\nZ9-Trefoil-Vert',
                'Vmode11\nZ10-Trefoil-Oblique', 'Vmode12\nZ11-Spherical']


## Full Nightly Report

In [ ]:
import os, sys 
os.environ["no_proxy"] += ",.consdb"

db = AOSDatabase(day_obs=day_obs, seq_min=seq_min, seq_max=seq_max)
await db.create(simplified=False)
table = db.table


In [ ]:
if len(table)>0:
    filtered_table = table[table['day_obs'] == day_obs]
    filtered_table['rotation_angle'] = filtered_table['rotation_angle'].astype(float)
    filtered_table['seq_num_corr'] = filtered_table['seq_num_corr'].astype(int)
    raw_filtered_table = table[table['day_obs'] == day_obs]
    filtered_table = filtered_table.select_dtypes(include="number")
    filtered_table['band'] = raw_filtered_table['band']
    filtered_table = filtered_table.groupby("seq").agg({col: 'first' if col == 'band' else 'mean' for col in filtered_table.columns})
    
    # -- AOS jumps
    aos_diff = filtered_table["aos_fwhm"].diff()
    jump_indices = filtered_table["seq"][aos_diff > 0.3].tolist()

    # -- States array and labels
    state_table = raw_filtered_table[["seq", "dof_state", "zernikes_fwhm", "lut_state"]]
    state_table = state_table.groupby('seq').mean()
    states_per_seq = state_table.dropna(subset=["dof_state", "zernikes_fwhm", "lut_state"])
    dof_state = np.vstack(states_per_seq["dof_state"].values)
    zernikes_fwhm = np.vstack(states_per_seq["zernikes_fwhm"].values)
    lut_state = np.vstack(states_per_seq["lut_state"].values)
    seqs = states_per_seq.index.values
else:
    print(f'No  data available for {day_obs},  seq_nums {seq_min}-{seq_max}')

# This cell recovers the ratios between the zernikes in arcseconds and in microns
# I know this is a hack, but it works for now.
index = 23
zernike_ratios = {}
for id_group, zk_group in enumerate(zk_groups):
    zk_labels = zk_group_labels[id_group]
    for zk_idx, i in enumerate(zk_group):
        #print(zk_idx, i, f"{zk_group_labels[id_group]}")
        ratio = raw_filtered_table['zernikes'][index][i] / raw_filtered_table['zernikes_fwhm'][index][i]
        zernike_ratios[zk_labels] = ratio


In [ ]:
if len(table)>0:
    fig = plt.figure(figsize=(31, 18))
    
    linewidth = 0.7
    # Top 5x4 GridSpec occupies the upper half
    gs_top = gridspec.GridSpec(
        nrows=6, ncols=4,
        width_ratios=[4, 2, 2.15, 2],
        height_ratios=[1]*6,
        hspace=0.0,
        wspace=0.26,
        top=0.97,
        bottom=0.54  # ends halfway down
    )
    
    # Bottom 5x4 GridSpec occupies the lower half
    gs_bot = gridspec.GridSpec(
        nrows=6, ncols=4,
        width_ratios=[4, 2, 2.15, 2],
        height_ratios=[1]*6,
        hspace=0.0,
        wspace=0.26,
        top=0.46,
        bottom=0.05
    )
    
    # -- Left column: Survey performance
    axes = []
    for i in range(6):
        ax = fig.add_subplot(gs_top[i, 0], sharex=axes[0] if i > 0 else None)
        axes.append(ax)
    axes[0].scatter(filtered_table['seq'], filtered_table['fwhm_zenith_500nm'], s=3, label='FWHM_Zenith_500nm')
    axes[0].scatter(filtered_table['seq'], filtered_table['donut_blur_fwhm'], s=3, label='Donut Blur FWHM')
    axes[0].scatter(filtered_table['seq'], filtered_table["psf_fwhm_95_05"], s=3, label="FWHM:sqrt(95^2-5^2)")
    ax0_ell = axes[0].twinx()
    ax0_ell.scatter(filtered_table['seq'], filtered_table["psf_ellipticity_median"], s=3, color='red', label="Ellipticity      ")
    ax0_ell.set_ylim(0, 0.8)
    ax0_ell.set_yticks([0.0, 0.1, 0.2])
    ax0_ell.set_ylabel("Ellipticity")
    ax0_ell.yaxis.set_label_coords(1.02, 0.65)
    h1, l1 = axes[0].get_legend_handles_labels()
    h2, l2 = ax0_ell.get_legend_handles_labels()
    ax0_ell.yaxis.set_label_coords(1.02, 0.65)
    axes[0].legend(h1 + h2, l1 + l2, fontsize=8, bbox_to_anchor=(1.0, 1.6), 
                   loc='upper right', markerscale=2)
    
    ymin0 = 0; ymax0 = 2.0
    axes[0].set_ylim(ymin0, ymax0)
    (xmin, xmax) = axes[0].get_xlim()
    axes[0].text(xmin, ymax0 * 1.1, "... Faults", fontsize=14, color='magenta')
    axes[0].set_ylabel('FWHM [arcsec]')
    axes[0].set_title(f'Delivered Seeing and System Variables')
    axes[1].scatter(filtered_table['seq'], filtered_table["aos_fwhm"], s=3, label='AOS FWHM')
    axes[2].scatter(filtered_table['seq'], filtered_table['elevation'], color='k', s=3)
    axes[3].scatter(filtered_table['seq'], filtered_table['azimuth'], color='k', s=3)
    axes[4].scatter(filtered_table['seq'], filtered_table['rotation_angle'], color='k', s=3)
    max_corr_lag = 10
    corr_lag = filtered_table['seq'] - filtered_table['seq_num_corr']
    corr_lag = np.clip(corr_lag, 0, max_corr_lag)
    axes[5].scatter(filtered_table['seq'], corr_lag, color='k', s=3)
    axes[5].minorticks_on()

    
    axes[0].set_ylabel('FWHM\n[arcsec]')
    axes[1].set_ylabel('AOS FWHM\n[arcsec]')
    axes[2].set_ylabel('Elevation\n[deg]')
    axes[3].set_ylabel('Azimuth\n[deg]')
    axes[4].set_ylabel('Rotator\n[deg]')
    axes[5].set_ylabel('Correction Lag\n[# of visits]')
    axes[5].set_xlabel('Sequence Number')
    
    for ax in axes[1::]:
        for x in jump_indices:
            ax.axvline(x=x, color='red', linestyle='--', linewidth=linewidth)
    # Add block boundaries and names
    axes[1].set_ylim(0.25, 1.25)
    axes[1].set_yticks([0.5, 1.0])
    blocks = find_blocks(table) # Find active blocks
    maxSeq = max(table['seq'].values)
    for i, changeSeq in enumerate(blocks['seq']):
        if i < len(blocks) - 1:
            changeWidth = blocks['seq'].iloc[i+1] - changeSeq
        else:
            changeWidth = maxSeq - changeSeq
        for ax in axes[0::]:
            ax.axvline(changeSeq, ls = '--', color='k', linewidth=linewidth, alpha=0.5)
        if changeWidth > 5:
            (ymin1, ymax1) = axes[1].get_ylim()
            ytext = ymin1 + (ymax1 - ymin1) * 0.5
            axes[1].text(int(changeSeq + changeWidth / 2 - 2), ytext, 
                         blocks['block_after'].iloc[i], fontsize=8, rotation=90, alpha=0.6)    

    # Now plot the faults
    faults = table[table['faults'].notnull()] 
    for k in range(len(faults)):
        for ax in axes[0::]:
            ax.axvline(faults['seq'].iloc[k], color='magenta', ls=':')

    for ax in axes:
        ax.grid(True, alpha=0.5)
        ax.tick_params(direction='in', which='both')

    for ax in axes[:-1]:
        ax.tick_params(labelbottom=False)

    for ax in axes:
        annotate_bands(filtered_table, ax)
    annotate_bands(filtered_table, axes[5], bottom_stripes=True)
    axes[5].legend(ncols=3)
    
    # Thermal gradients 
    axes[0] = fig.add_subplot(gs_bot[0:2, 0])
    axes[1] = fig.add_subplot(gs_bot[2:4, 0])
    axes[2] = fig.add_subplot(gs_bot[4:, 0])
    axes = [axes[0], axes[1], axes[2]]
    plot_m1m3_gradients(filtered_table, axes[0])
    min_temp = 1000.0; max_temp = -1000
    for temp in ['cam_air_temp', 'm2_air_temp', 'm1m3_air_temp', 'outside_temp']:
        this_max = np.max(filtered_table[temp])
        this_min = np.min(filtered_table[temp])
        if  this_max > max_temp:
            max_temp = this_max
        if  this_min < min_temp:
            min_temp = this_min
    axes[1].scatter(filtered_table['seq'], filtered_table['cam_air_temp'], color="C0", s=3, label="ESS:111 camera air Temperature")
    axes[1].scatter(filtered_table['seq'], filtered_table['m2_air_temp'], color="C1", s=3, label="ESS:112 M2 air Temperature")
    axes[1].scatter(filtered_table['seq'], filtered_table['m1m3_air_temp'], color="C2", s=3, label="ESS:113 M1M3 air Temperature")
    axes[1].scatter(filtered_table['seq'], filtered_table['outside_temp'], color="C6", s=3, label="ESS:301 Outside Temperature")
    leg4 = axes[1].legend(loc='upper right', ncols=2, markerscale=2)
    leg4.get_frame().set_linewidth(0)
    leg4.get_frame().set_edgecolor("none")
    leg4.get_frame().set_facecolor("none")

    axes[1].set_ylim(min_temp * 0.95, max_temp * 1.1)
    
    axes[2].scatter(filtered_table['seq'], filtered_table['m2_delta_t'], color='C0', s=3, label='M2-M1M3')
    axes[2].scatter(filtered_table['seq'], filtered_table['cam_m1m3_delta_t'], color='C1', s=3, label='Cam-M1M3')
    axes[2].scatter(filtered_table['seq'], filtered_table['dome_delta_t'], color='C2', s=3, label='Out-M1M3')
    leg3 = axes[2].legend(loc='upper right', ncols=3, markerscale=2, framealpha=0.5)
    leg3.get_frame().set_linewidth(0)
    leg3.get_frame().set_edgecolor("none")
    leg3.get_frame().set_facecolor("none")

    
    axes[1].set_ylabel('Temps\n[C]')
    axes[2].set_ylabel('Temp DIffs\n[C]')
    axes[2].set_xlabel('Sequence Number')
    
    for ax in axes:
        for x in jump_indices:
            ax.axvline(x=x, color='red', linestyle='--', linewidth=linewidth)
    for ax in axes[:-1]:
        ax.tick_params(labelbottom=False)
    for ax in axes:
        ax.grid(True, alpha=0.5)
        ax.tick_params(direction='in', which='both')
    axes[0].set_title(f'M1M3 Thermal Gradients', y=1.02)
    
    # -- Right column: DoF plots with shared x-axis (not y)
    axes = [fig.add_subplot(gs_top[i, 1]) for i in range(6)]
    for id_group, (ax, zk_group) in enumerate(zip(axes, zk_groups)):
        zk_labels = zk_group_labels[id_group]
        for zk_idx, i in enumerate(zk_group):
            # This loop is for setting the limits
            # This algorithm was recommended by Aaron Roodman
            vals = zernikes_fwhm[:, i]
            if zk_idx == 0:
                combined_vals = vals
            else:
                combined_vals = np.concatenate((combined_vals, vals))
            ymin = np.percentile(combined_vals, 1)
            ymax = np.percentile(combined_vals, 99)
            if zk_labels == 'Z4':
                ymax = min(0.2, ymax)
            if ymin < 0:
                ymin *= 1.25
            else:
                ymin *= 0.75
            if ymax > 0:
                ymax *= 1.25
            else:
                ymax *= 0.75
        #print(f"{zk_group_labels[id_group]}, ymin = {ymin:.4f}, ymax = {ymax:.4f}, Micron/Arcsec ratio = {zernike_ratios[zk_labels]:.4f}") 
        ymin_microns = zernike_ratios[zk_labels] * ymin
        ymax_microns = zernike_ratios[zk_labels] * ymax
        for zk_idx, i in enumerate(zk_group):
            # This loop does the scatter plots
            if len(zk_group) == 1:
                zk_label = ''
            if len(zk_group) == 2:
                zk_label = zk_labels.split('/')[zk_idx].strip()
            vals = zernikes_fwhm[:, i]
            color = 'blue' if zk_idx == 0 else 'orange' if zk_idx == 1 else None
            clip_vals = np.clip(vals, ymin, ymax)
            ax.scatter(seqs, clip_vals, s=5, color=color, label=zk_label)
        ax.set_ylabel(f"{zk_group_labels[id_group]}\n[arcsec]")
        if zk_group_labels[id_group] == 'Z4':
            # This line is showing the recently added focus offset
            ax.axhline(-0.15, ls='--', color='green')
        ax.grid(True, alpha=0.5)
        ax.tick_params(direction="in")
        ax.set_ylim(ymin, ymax)
        ax2 = ax.twinx()
        ax2.set_ylim(ymin_microns, ymax_microns)
        ax2.set_ylabel("Microns")
        leg1 = ax.legend(bbox_to_anchor=(0.2, 1.05), loc='upper left',
                         ncol=2, markerscale=2, frameon=False)

    annotate_bands(filtered_table, axes[0])
    
    for ax in axes[:-1]:
        ax.tick_params(labelbottom=False)
        ax.grid(True, alpha=0.5)
    for ax in axes:
        for x in jump_indices:
            ax.axvline(x=x, color='red', linestyle='--', linewidth=linewidth)
    axes[-1].set_xlabel("Sequence Number")
    axes[0].set_title(f'Optical Aberrations (average over FOV)')
    
    
    # -- Center column: DoF plots with shared x-axis (not y)
    axes = [fig.add_subplot(gs_bot[i, 1]) for i in range(6)]
    for id_group, (ax, dof_group) in enumerate(zip(axes, groups)):
        for i in dof_group:
            vals = dof_state[:, i]#(lut_state[:, i] + dof_state[:, i])
            ax.scatter(seqs, vals, label=f"{labels[i]}", s=3)
        ax.set_ylabel(group_labels[id_group])
        ax.grid(True, alpha=0.5)
        ax.tick_params(direction="in")
        leg2 = ax.legend(bbox_to_anchor=(1.28, 0.5), loc='center right', markerscale=3)
        leg2.get_frame().set_linewidth(0)
        leg2.get_frame().set_edgecolor("none")
        leg2.get_frame().set_facecolor("none")

    annotate_bands(filtered_table, axes[0])
    annotate_bands(filtered_table, axes[1])
    
    for ax in axes[:-1]:
        ax.tick_params(labelbottom=False)
        ax.grid(True, alpha=0.5)
    for ax in axes:
        for x in jump_indices:
            ax.axvline(x=x, color='red', linestyle='--', linewidth=linewidth)
    axes[0].set_title(f'Hexapods State (Trim only)')
    axes[-1].set_xlabel("Sequence Number")


    # -- Right column: System contribution plots
    ax1 = fig.add_subplot(gs_top[3:, 2])
    ax0 = fig.add_subplot(gs_top[:3, 2], sharex=ax1)
    ax3 = fig.add_subplot(gs_bot[3:, 2])
    ax2 = fig.add_subplot(gs_bot[:3, 2], sharex=ax3)
    ax5 = fig.add_subplot(gs_bot[3:, 3])
    ax4 = fig.add_subplot(gs_bot[:3, 3], sharex=ax5)
    axes = [ax0, ax1, ax2, ax3, ax4, ax5]

    axes[0].set_xlim(0.5, 2.0)
    axes[0].set_ylim(0.5, 2.0)
    axes[0].set_ylabel("Donut blur [arcsec]")
    axes[0].set_title("Image Quality Metrics")
    axes[0].set_aspect(1.0)
    axes[0].tick_params(labelbottom=False)
    axes[0].grid(alpha=0.2)
    xs = np.linspace(0.5, 2.0, 100)
    ys = np.sqrt(xs**2 - 0.40**2)
    axes[0].plot(xs, xs, ls='--', color='red')
    axes[0].plot(xs, ys, ls='--', color='green')
    axes[0].text(0.1, 0.9, 'FWHM=Blur',
                  horizontalalignment='left', verticalalignment='center', 
                   transform=axes[0].transAxes, color='red')
    axes[0].text(0.1, 0.85, 'FWHM=sqrt(Blur**2 + 0.4**2)',
                  horizontalalignment='left', verticalalignment='center', 
                   transform=axes[0].transAxes, color='green')

    donut_blur = filtered_table['donut_blur_fwhm'].values
    ax0_histy = axes[0].inset_axes([1.0, 0, 0.5, 1], sharey=axes[0])
    ax0_histy.hist(donut_blur, bins=20, histtype='step', 
                   range=(0.5, 2.0), lw=2, orientation='horizontal')
    ax0_histy.yaxis.tick_right()
    ax0_histy.set_xticks([])
    ax0_histy.set_xticklabels([])
    ax0_histy.grid(alpha=0.2)
    ax0_histy.text(0.5, 0.8, f'Median = {np.nanmedian(donut_blur):.2f}"',
                  horizontalalignment='center', verticalalignment='center', 
                   transform=ax0_histy.transAxes)

    fwhm = filtered_table['psf_fwhm'].values
    ax0_histx = axes[0].inset_axes([0, 1.0, 1, 0.25], sharex=axes[0])
    ax0_histx.hist(fwhm, bins=20, histtype='step', lw=2, orientation='vertical')
    ax0_histx.grid(alpha=0.2)
    ax0_histx.text(0.8, 0.8, f'Median = {np.nanmedian(fwhm):.2f}"',
                  horizontalalignment='center', verticalalignment='center', 
                   transform=ax0_histx.transAxes)

    x  = filtered_table["psf_fwhm"].values
    y  = filtered_table["psf_ellipticity_median"].values
    xerr = filtered_table["psf_fwhm_sigma"].values
    yerr = filtered_table["psf_ellipticity_sigma"].values
    for band in band_colors.keys():
        band_table = filtered_table[filtered_table['band'].str[0] == band]
        axes[0].scatter(band_table['psf_fwhm'], band_table['donut_blur_fwhm'], 
                        s=8, color=band_colors[band])
        x_band  = band_table["psf_fwhm"].values
        y_band  = band_table["psf_ellipticity_median"].values
        xerr_band = band_table["psf_fwhm_sigma"].values
        yerr_band = band_table["psf_ellipticity_sigma"].values
        axes[1].errorbar(
            x_band, y_band,
            xerr=xerr_band,
            yerr=yerr_band,
            fmt='o', markersize=3,
            elinewidth=0.1, capsize=1,
            color=band_colors[band], alpha=0.7
        )
    axes[1].set_ylabel("Ellipticity")
    axes[1].set_xlabel("FWHM [arcsec]")
    axes[1].set_xticks([0.6, 1.0, 1.4, 1.8, 2.0])
    axes[1].set_box_aspect(1.0)
    axes[1].grid(alpha=0.2)

    ax1_histy = axes[1].inset_axes([1.0, 0, 0.5, 1], sharey=axes[1])
    ax1_histy.hist(y, bins=20, histtype='step', lw=2, orientation='horizontal')
    ax1_histy.yaxis.tick_right()
    ax1_histy.grid(alpha=0.2)
    ax1_histy.text(0.5, 0.8, f'Median = {np.nanmedian(y):.3f}"',
                  horizontalalignment='center', verticalalignment='center', 
                   transform=ax1_histy.transAxes)

    camera_contrib = 0.207 # From lab measurements at SLAC
    sys_3  = np.sqrt(filtered_table["psf_fwhm"].values**2 - (filtered_table["donut_blur_fwhm"].values**2 - camera_contrib**2))
    y  = filtered_table["psf_ellipticity_median"].values
    xerr = filtered_table["psf_fwhm_sigma"].values
    in_spec = [[a,b] for a,b in zip(sys_3,y) if (a < 0.45) and (b < 0.1)]
    pass_2 = f"\n{(len(in_spec) / len(sys_3) * 100.0):.1f}% in green region"
    axes[2].scatter(sys_3, y, s=4, color='tab:orange')
    axes[2].set_ylabel("Ellipticity")
    axes[2].set_box_aspect(1.0)
    axes[2].tick_params(labelbottom=False)
    axes[2].grid(alpha=0.2)
    axes[2].set_title("System Contribution Metrics")
    axes[2].set_ylim(top = 0.5)

    x  = filtered_table["psf_fwhm_95_05"]
    y  = filtered_table["psf_ellipticity_median"].values
    xerr = filtered_table["psf_fwhm_sigma"].values
    in_spec = [[a,b] for a,b in zip(x,y) if (a < 0.45) and (b < 0.1)]
    pass_3 = f"\n{(len(in_spec) / len(x) * 100.0):.1f}% in green region"
    axes[3].scatter(x, y, s=4, color='tab:red')
    axes[3].set_ylabel("Ellipticity")
    axes[3].set_xlabel("[arcsec]")
    axes[3].set_box_aspect(1.0)
    axes[3].grid(alpha=0.2)
    axes[3].set_ylim(top = 0.5)

    sys_2 = np.sqrt(filtered_table["aos_fwhm"].values**2 + camera_contrib**2)
    axes[4].hist(sys_2, bins=30, range=(0.0, 1.0), histtype='step', lw=2, color='tab:blue', label='sqrt(AOS^2+Cam^2)')
    axes[4].hist(sys_3, bins=30, range=(0.0, 1.0), histtype='step', lw=2, color='tab:orange', label='sqrt(FWHM^2-(Blur^2-Cam^2))')
    axes[4].hist(filtered_table["psf_fwhm_95_05"], bins=30, range=(0.0, 1.0), histtype='step', lw=2, color='tab:red', label='FWHM:sqrt(95^2-5^2)')
    axes[4].set_xlim(0.0, 1.0)
    axes[4].set_box_aspect(1.0)
    axes[4].tick_params(labelbottom=False)
    axes[4].axvline(0.45, ls='--', color='black', alpha=0.5)
    axes[4].text(0.2, 0.90, f'Median = {np.nanmedian(sys_2):.2f}"',
              horizontalalignment='center', verticalalignment='center', 
              transform=axes[4].transAxes, color='tab:blue')
    axes[4].text(0.2, 0.85, f'Median = {np.nanmedian(sys_3):.2f}"',
              horizontalalignment='center', verticalalignment='center', 
              transform=axes[4].transAxes, color='tab:orange')
    axes[4].text(0.2, 0.80, f'Median = {np.nanmedian(filtered_table["psf_fwhm_95_05"].values):.2f}"',
              horizontalalignment='center', verticalalignment='center', 
              transform=axes[4].transAxes, color='tab:red')

    legend_patches = [
        Patch(facecolor='tab:blue',   edgecolor='none', label='sqrt(AOS^2 + Cam^2)'),
        Patch(facecolor='tab:orange',    edgecolor='none', label='sqrt(FWHM^2 - (Blur^2 - Cam^2))'),
        Patch(facecolor='tab:red',    edgecolor='none', label='FWHM: sqrt(95^2 - 5^2)'),

    ]
    
    axes[4].legend(
        handles=legend_patches,
        loc='upper center',
        bbox_to_anchor=(0.4, 1.2),
        frameon=False,
        borderaxespad=0.,
    )
    shift = -0.035
    pos = axes[4].get_position()
    axes[4].set_position([pos.x0 + shift, pos.y0, pos.width, pos.height])
    axes[4].grid(alpha=0.2)

    x  = sys_2
    y  = filtered_table["psf_ellipticity_median"].values
    xerr = filtered_table["psf_fwhm_sigma"].values
    in_spec = [[a,b] for a,b in zip(x,y) if (a < 0.45) and (b < 0.1)]
    pass_5 = f"\n{(len(in_spec) / len(x) * 100.0):.1f}% in green region"
    axes[5].scatter(x, y, s=4, color='tab:blue')
    axes[5].set_xlabel("[arcsec]")
    axes[5].set_box_aspect(1.0)
    shift = -0.035
    pos = axes[5].get_position()
    axes[5].set_position([pos.x0 + shift, pos.y0, pos.width, pos.height])
    axes[5].grid(alpha=0.2)
    axes[5].set_ylim(top = 0.5)

    titles = ["sqrt(FWHM^2-(Blur^2-Cam^2))", "FWHM:sqrt(95^2-5^2)", "sqrt(AOS^2+Cam^2)"]
    passes = [pass_2, pass_3, pass_5]
    for ax, t, p in zip([axes[2], axes[3], axes[5]], titles, passes):
        xmin, xmax = ax.get_xlim()
        ymin, ymax = ax.get_ylim()
    
        x_hi = min(0.45, xmax)
        y_hi = min(0.1, ymax)
    
        ax.fill_between(
            [xmin, x_hi],
            ymin, y_hi,
            color=(0.45, 0.85, 0.45),
            alpha=0.15,
            zorder=0
        )
        ax.set_xlim(xmin, xmax)
        ax.set_ylim(ymin, ymax)

        ax.text(
            (xmin + xmax) / 2,
            0.9*ymax,
            t + p,
            ha='center', va='center',
            color='black',
            alpha=0.8,
            zorder=2
        )

    my_suptitle = fig.suptitle("Survey Mode Performance and AOS DoFs – Day " + str(day_obs), fontsize=18, x=0.4, y=1.03)
else:
    print('No data to plot')
fig.savefig(f"/home/c/cslage/u/MTAOS/times_square_reports/Full_Night_Report_{day_obs}.png", 
            bbox_inches='tight', pad_inches=1.2, bbox_extra_artists=[my_suptitle])

## Detailed Nightly Report with full DOFs and Vmodes

In [ ]:
if make_detail_plot:
    db = AOSDatabase(day_obs=day_obs, seq_min=detailed_seq_min, seq_max=detailed_seq_max)
    await db.create(simplified=False)
    table = db.table


In [ ]:
if make_detail_plot:
    from lsst.ts.ofc.state_estimator import StateEstimator
    from lsst.ts.ofc import OFCData
    ofcData = OFCData()
    ofcData.configure_controller()
    await ofcData.configure_instrument('lsst')
    se = StateEstimator(ofcData)
    
    if len(table)>0:
        filtered_table = table[table['day_obs'] == day_obs]
        filtered_table['rotation_angle'] = filtered_table['rotation_angle'].astype(float)
        filtered_table['seq_num_corr'] = filtered_table['seq_num_corr'].astype(int)
        raw_filtered_table = table[table['day_obs'] == day_obs]
        filtered_table = filtered_table.select_dtypes(include="number")
        filtered_table['band'] = raw_filtered_table['band']
        filtered_table = filtered_table.groupby("seq").agg({col: 'first' if col == 'band' else 'mean' for col in filtered_table.columns})
        
        # -- AOS jumps
        aos_diff = filtered_table["aos_fwhm"].diff()
        jump_indices = filtered_table["seq"][aos_diff > 0.3].tolist()
    
        # -- States array and labels
        state_table = raw_filtered_table[["seq", "dof_state", "zernikes_fwhm", "lut_state"]]
        state_table = state_table.groupby('seq').mean()
        states_per_seq = state_table.dropna(subset=["dof_state", "zernikes_fwhm", "lut_state"])
        dof_state = np.vstack(states_per_seq["dof_state"].values)
        zernikes_fwhm = np.vstack(states_per_seq["zernikes_fwhm"].values)
        lut_state = np.vstack(states_per_seq["lut_state"].values)
        seqs = states_per_seq.index.values
    
        # Get Zernike ratios
        index = 23
        zernike_ratios = {}
        for id_group, zk_group in enumerate(zk_groups):
            zk_labels = zk_group_labels[id_group]
            for zk_idx, i in enumerate(zk_group):
                ratio = raw_filtered_table['zernikes'][index][i] / raw_filtered_table['zernikes_fwhm'][index][i]
                zernike_ratios[zk_labels] = ratio
    
        # get the vmodes
        vmodes = []
        for i in range(len(dof_state)):
            vmodes.append(se.get_vmodes_from_dofs(dof_state[i])[0:12])
        vmodes = np.array(vmodes)
    
    else:
        print(f'No  data available for {day_obs},  seq_nums {detailed_seq_min}-{detailed_seq_max}')

In [ ]:
if make_detail_plot:
    if len(table)>0:
        fig = plt.figure(figsize=(31, 18))
        
        linewidth = 0.7
        # Top 5x4 GridSpec occupies the upper half
        gs_top = gridspec.GridSpec(
            nrows=6, ncols=4,
            width_ratios=[4, 2, 2.15, 2],
            height_ratios=[1]*6,
            hspace=0.0,
            wspace=0.38,
            top=0.97,
            bottom=0.54  # ends halfway down
        )
        
        # Bottom 5x4 GridSpec occupies the lower half
        gs_bot = gridspec.GridSpec(
            nrows=6, ncols=4,
            width_ratios=[4, 2, 2.15, 2],
            height_ratios=[1]*6,
            hspace=0.0,
            wspace=0.38,
            top=0.46,
            bottom=0.05
        )
        
        # -- Left column: Survey performance
        axes = []
        for i in range(6):
            ax = fig.add_subplot(gs_top[i, 0], sharex=axes[0] if i > 0 else None)
            axes.append(ax)
        axes[0].scatter(filtered_table['seq'], filtered_table['fwhm_zenith_500nm'], s=9, label='FWHM_Zenith_500nm')
        axes[0].scatter(filtered_table['seq'], filtered_table['donut_blur_fwhm'], s=9, label='Donut Blur FWHM')
        axes[0].scatter(filtered_table['seq'], filtered_table["psf_fwhm_95_05"], s=9, label="FWHM:sqrt(95^2-5^2)")
        ax0_ell = axes[0].twinx()
        ax0_ell.scatter(filtered_table['seq'], filtered_table["psf_ellipticity_median"], s=9, color='red', label="Ellipticity      ")
        ax0_ell.set_ylim(0, 0.8)
        ax0_ell.set_yticks([0.0, 0.1, 0.2])
        ax0_ell.set_ylabel("Ellipticity")
        ax0_ell.yaxis.set_label_coords(1.02, 0.65)
        h1, l1 = axes[0].get_legend_handles_labels()
        h2, l2 = ax0_ell.get_legend_handles_labels()
        ax0_ell.yaxis.set_label_coords(1.02, 0.65)
        axes[0].legend(h1 + h2, l1 + l2, fontsize=8, bbox_to_anchor=(1.0, 1.6), 
                       loc='upper right', markerscale=2)
        
        ymin0 = 0; ymax0 = 2.0
        axes[0].set_ylim(ymin0, ymax0)
        (xmin, xmax) = axes[0].get_xlim()
        axes[0].text(xmin, ymax0 * 1.1, "... Faults", fontsize=14, color='magenta')
        axes[0].set_ylabel('FWHM [arcsec]')
        axes[0].set_title(f'Delivered Seeing and System Variables')
        axes[1].scatter(filtered_table['seq'], filtered_table["aos_fwhm"], s=9, label='AOS FWHM')
        axes[2].scatter(filtered_table['seq'], filtered_table['elevation'], color='k', s=9)
        axes[3].scatter(filtered_table['seq'], filtered_table['azimuth'], color='k', s=9)
        axes[4].scatter(filtered_table['seq'], filtered_table['rotation_angle'], color='k', s=9)
        max_corr_lag = 10
        corr_lag = filtered_table['seq'] - filtered_table['seq_num_corr']
        corr_lag = np.clip(corr_lag, 0, max_corr_lag)
        axes[5].scatter(filtered_table['seq'], corr_lag, color='k', s=9)
        axes[5].minorticks_on()
    
        
        axes[0].set_ylabel('FWHM\n[arcsec]')
        axes[1].set_ylabel('AOS FWHM\n[arcsec]')
        axes[2].set_ylabel('Elevation\n[deg]')
        axes[3].set_ylabel('Azimuth\n[deg]')
        axes[4].set_ylabel('Rotator\n[deg]')
        axes[5].set_ylabel('Correction Lag\n[# of visits]')
        axes[5].set_xlabel('Sequence Number')
        
        for ax in axes[1::]:
            for x in jump_indices:
                ax.axvline(x=x, color='red', linestyle='--', linewidth=linewidth)
        # Add block boundaries and names
        axes[1].set_ylim(0.25, 1.25)
        axes[1].set_yticks([0.5, 1.0])
        blocks = find_blocks(table) # Find active blocks
        maxSeq = max(table['seq'].values)
        for i, changeSeq in enumerate(blocks['seq']):
            if i < len(blocks) - 1:
                changeWidth = blocks['seq'].iloc[i+1] - changeSeq
            else:
                changeWidth = maxSeq - changeSeq
            for ax in axes[0::]:
                ax.axvline(changeSeq, ls = '--', color='k', linewidth=linewidth, alpha=0.5)
            if changeWidth > 5:
                (ymin1, ymax1) = axes[1].get_ylim()
                ytext = ymin1 + (ymax1 - ymin1) * 0.5
                axes[1].text(int(changeSeq + changeWidth / 2 - 2), ytext, 
                             blocks['block_after'].iloc[i], fontsize=8, rotation=90, alpha=0.6)    
    
        # Now plot the faults
        faults = table[table['faults'].notnull()] 
        for k in range(len(faults)):
            for ax in axes[0::]:
                ax.axvline(faults['seq'].iloc[k], color='magenta', ls=':')
    
        for ax in axes:
            ax.grid(True, alpha=0.5)
            ax.tick_params(direction='in', which='both')
    
        for ax in axes[:-1]:
            ax.tick_params(labelbottom=False)
    
        for ax in axes:
            annotate_bands(filtered_table, ax)
        annotate_bands(filtered_table, axes[5], bottom_stripes=True)
        axes[5].legend(ncols=9)
        
        # Thermal gradients 
        axes[0] = fig.add_subplot(gs_bot[0:2, 0])
        axes[1] = fig.add_subplot(gs_bot[2:4, 0])
        axes[2] = fig.add_subplot(gs_bot[4:, 0])
        axes = [axes[0], axes[1], axes[2]]
        plot_m1m3_gradients(filtered_table, axes[0])
        min_temp = 1000.0; max_temp = -1000
        for temp in ['cam_air_temp', 'm2_air_temp', 'm1m3_air_temp', 'outside_temp']:
            this_max = np.max(filtered_table[temp])
            this_min = np.min(filtered_table[temp])
            if  this_max > max_temp:
                max_temp = this_max
            if  this_min < min_temp:
                min_temp = this_min
        axes[1].scatter(filtered_table['seq'], filtered_table['cam_air_temp'], color="C0", s=9, label="ESS:111 camera air Temperature")
        axes[1].scatter(filtered_table['seq'], filtered_table['m2_air_temp'], color="C1", s=9, label="ESS:112 M2 air Temperature")
        axes[1].scatter(filtered_table['seq'], filtered_table['m1m3_air_temp'], color="C2", s=9, label="ESS:113 M1M3 air Temperature")
        axes[1].scatter(filtered_table['seq'], filtered_table['outside_temp'], color="C6", s=9, label="ESS:301 Outside Temperature")
        leg4 = axes[1].legend(loc='upper right', ncols=2, markerscale=2)
        leg4.get_frame().set_linewidth(0)
        leg4.get_frame().set_edgecolor("none")
        leg4.get_frame().set_facecolor("none")
    
        axes[1].set_ylim(min_temp * 0.95, max_temp * 1.1)
        
        axes[2].scatter(filtered_table['seq'], filtered_table['m2_delta_t'], color='C0', s=9, label='M2-M1M3')
        axes[2].scatter(filtered_table['seq'], filtered_table['cam_m1m3_delta_t'], color='C1', s=9, label='Cam-M1M3')
        axes[2].scatter(filtered_table['seq'], filtered_table['dome_delta_t'], color='C2', s=9, label='Out-M1M3')
        leg3 = axes[2].legend(loc='upper right', ncols=3, markerscale=2, framealpha=0.5)
        leg3.get_frame().set_linewidth(0)
        leg3.get_frame().set_edgecolor("none")
        leg3.get_frame().set_facecolor("none")
    
        
        axes[1].set_ylabel('Temps\n[C]')
        axes[2].set_ylabel('Temp DIffs\n[C]')
        axes[2].set_xlabel('Sequence Number')
        
        for ax in axes:
            for x in jump_indices:
                ax.axvline(x=x, color='red', linestyle='--', linewidth=linewidth)
        for ax in axes[:-1]:
            ax.tick_params(labelbottom=False)
        for ax in axes:
            ax.grid(True, alpha=0.5)
            ax.tick_params(direction='in', which='both')
        axes[0].set_title(f'M1M3 Thermal Gradients', y=1.02)
    
        # -- Right column: DoF plots with shared x-axis (not y)
        axes = [fig.add_subplot(gs_top[i, 1]) for i in range(6)]
        for id_group, (ax, zk_group) in enumerate(zip(axes, zk_groups)):
            zk_labels = zk_group_labels[id_group]
            for zk_idx, i in enumerate(zk_group):
                # This loop is for setting the limits
                vals = zernikes_fwhm[:, i]
                if zk_idx == 0:
                    combined_vals = vals
                else:
                    combined_vals = np.concatenate((combined_vals, vals))
                ymin = np.percentile(combined_vals, 1)
                ymax = np.percentile(combined_vals, 99)
    
                if zk_labels == 'Z4':
                    ymax = min(0.20, ymax)
    
                if ymin < 0:
                    ymin *= 1.25
                else:
                    ymin *= 0.75
                if ymax > 0:
                    ymax *= 1.25
                else:
                    ymax *= 0.75
            ymin_microns = zernike_ratios[zk_labels] * ymin
            ymax_microns = zernike_ratios[zk_labels] * ymax
            for zk_idx, i in enumerate(zk_group):
                # This loop does the scatter plots
                if len(zk_group) == 1:
                    zk_label = ''
                if len(zk_group) == 2:
                    zk_label = zk_labels.split('/')[zk_idx].strip()
                vals = zernikes_fwhm[:, i]
                color = 'C0' if zk_idx == 0 else 'C1' if zk_idx == 1 else None
                clip_vals = np.clip(vals, ymin, ymax)
                ax.scatter(seqs, clip_vals, s=9, color=color, label=zk_label)
            ax.set_ylabel(f"{zk_group_labels[id_group]}\n[arcsec]")
            if zk_group_labels[id_group] == 'Z4':
                # This line is showing the recently added focus offset
                ax.axhline(-0.15, ls='--', color='green')
            ax.grid(True, alpha=0.5)
            ax.tick_params(direction="in")
            ax.set_ylim(ymin, ymax)
            ax2 = ax.twinx()
            ax2.set_ylim(ymin_microns, ymax_microns)
            ax2.set_ylabel("Microns")
            leg1 = ax.legend(bbox_to_anchor=(0.2, 1.05), loc='upper left',
                             ncol=2, markerscale=2, frameon=False)
    
        annotate_bands(filtered_table, axes[0])
        
        for ax in axes[:-1]:
            ax.tick_params(labelbottom=False)
            ax.grid(True, alpha=0.5)
        for ax in axes:
            for x in jump_indices:
                ax.axvline(x=x, color='red', linestyle='--', linewidth=linewidth)
        axes[-1].set_xlabel("Sequence Number")
        axes[0].set_title(f'Optical Aberrations (average over FOV)')
        
        
        # -- Center column: DoF plots with shared x-axis (not y)
        axes = [fig.add_subplot(gs_bot[i, 1]) for i in range(6)]
        for id_group, (ax, dof_group) in enumerate(zip(axes, groups)):
            for i in dof_group:
                vals = dof_state[:, i]#(lut_state[:, i] + dof_state[:, i])
                ax.scatter(seqs, vals, label=f"{labels[i]}", s=9)
            ax.set_ylabel(group_labels[id_group])
            ax.grid(True, alpha=0.5)
            ax.tick_params(direction="in")
            leg2 = ax.legend(bbox_to_anchor=(1.28, 0.5), loc='center right', markerscale=3)
            leg2.get_frame().set_linewidth(0)
            leg2.get_frame().set_edgecolor("none")
            leg2.get_frame().set_facecolor("none")
    
        annotate_bands(filtered_table, axes[0])
        annotate_bands(filtered_table, axes[1])
        
        for ax in axes[:-1]:
            ax.tick_params(labelbottom=False)
            ax.grid(True, alpha=0.5)
        for ax in axes:
            for x in jump_indices:
                ax.axvline(x=x, color='red', linestyle='--', linewidth=linewidth)
        axes[0].set_title(f'Hexapods State (Trim only)')
        axes[-1].set_xlabel("Sequence Number")
    
    
        # -- Next column: Mirror mode plots with shared x-axis (not y)
        axes1 = [fig.add_subplot(gs_top[i, 2]) for i in range(6)]
        axes2 = [fig.add_subplot(gs_bot[i, 2]) for i in range(6)]
        axes = axes1 + axes2
        indices = list(range(10, 17)) + list(range(30,35))
        for n, i in enumerate(indices):
            vals = dof_state[:, i]
            axes[n].scatter(seqs, vals, s=11)
            axes[n].set_ylabel(all_labels[i])
        for ax in axes:
            ax.tick_params(labelbottom=False)
            ax.grid(True, alpha=0.5)
        axes[0].set_title(f'Miror DOFs (Trim only)')
        axes[5].tick_params(labelbottom=True)
        axes[5].set_xlabel("Sequence Number")
        axes[11].tick_params(labelbottom=True)
        axes[11].set_xlabel("Sequence Number")
    
    
        # -- Right column: Vmode plots with shared x-axis (not y)
        axes1 = [fig.add_subplot(gs_top[i, 3]) for i in range(6)]
        axes2 = [fig.add_subplot(gs_bot[i, 3]) for i in range(6)]
        axes = axes1 + axes2
        for i in range(12):
            axes[i].scatter(seqs, vmodes[:,i], s=11)
            axes[i].set_ylabel(vmode_labels[i])
        for ax in axes:
            ax.tick_params(labelbottom=False)
            ax.grid(True, alpha=0.5)
        axes[0].set_title(f'Vmodes')
        axes[5].tick_params(labelbottom=True)
        axes[5].set_xlabel("Sequence Number")
        axes[11].tick_params(labelbottom=True)
        axes[11].set_xlabel("Sequence Number")
    
    
        my_suptitle = fig.suptitle("Survey Mode Performance and AOS DoFs – Day " + str(day_obs), fontsize=18, x=0.4, y=1.03)
    else:
        print('No data to plot')


In [ ]:
index1 = 16
index2 = 17

indices = [0,3,4,5,8,9] + list(range(10, 17)) + list(range(30,35))
print(f"DOF name            DOF-{seqs[index1]}          DOF-{seqs[index2]}            Diff      % change")
for i in indices:
    diff = (dof_state[index2,i] - dof_state[index1,i])
    pct = diff / dof_state[index1,i] * 100.0
    if i < 10:
        label=all_labels[i] + " "
    else:
        label=all_labels[i]
    print(f"{label}\t \t {dof_state[index1,i]:10.4f}\t  {dof_state[index2,i]:10.4f}\t   {diff:10.4f} \t    {pct:4.1f}")


In [ ]:
table.columns